# Госпитальные_исходы: модели, Optuna и контролируемые эксперименты

Переносимая версия расчётного кода. Выводы очищены. Результаты записываются только в `runs/vascular`. Исходные снимки в `results/` не изменяются. Полный поиск в 02 требует существенного времени. См. README и docs/REPRODUCIBILITY.md.


In [ ]:
from pathlib import Path
import sys, os, json, hashlib, platform, importlib, datetime, tempfile
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "clinical-mortality-mpl"))
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "repository_support.py").exists())
if str(PROJECT_ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
from repository_support import initialize
OUT = initialize("vascular")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE_SEED = 2026
VERSIONS = {m: importlib.import_module(m).__version__ for m in ["numpy", "pandas", "sklearn", "optuna", "shap", "catboost", "xgboost", "nbclient"]}
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def savejson(obj, name): (OUT / name).write_text(json.dumps(obj, ensure_ascii=False, indent=2, default=str))
def figsave(name):
    plt.tight_layout(); plt.savefig(OUT / (name + ".png"), dpi=180, bbox_inches="tight"); plt.savefig(OUT / (name + ".svg"), bbox_inches="tight"); plt.show(); plt.close()
print("Working directory:", OUT, "Python:", platform.python_version(), "seed:", BASE_SEED, VERSIONS)


In [ ]:
import time, warnings, joblib, optuna, sqlite3, math
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, confusion_matrix, roc_curve, precision_score, recall_score, f1_score, accuracy_score
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from threadpoolctl import threadpool_limits
threadpool_limits(2)
optuna.logging.set_verbosity(optuna.logging.WARNING)
meta=json.loads((OUT/"01_конфигурация.json").read_text())
assert sha(OUT/"01_данные.csv") == meta["data_sha256"]
# Public data integrity is checked through metadata/data SHA-256.
D=pd.read_csv(OUT/"01_данные.csv")
FEATURES=meta["features"]
X=D[FEATURES];y=D.hospital_death.to_numpy();groups=D.group_id.to_numpy()
FAMILIES=["LR","RF","CatBoost","XGBoost"]

STRATEGIES=['all_real','under_10','under_5']
BASE={"LR":{"C":1.0},"RF":{"n_estimators":500,"max_depth":5,"min_samples_leaf":1,"max_features":"sqrt"},"CatBoost":{"iterations":300,"depth":4,"learning_rate":0.05,"l2_leaf_reg":3.0},"XGBoost":{"n_estimators":300,"max_depth":3,"learning_rate":0.05,"min_child_weight":1.0,"subsample":1.0,"colsample_bytree":1.0,"reg_lambda":1.0,"reg_alpha":0.0}}
CONFIG=dict(version='hospital_updated55_serial_fixed_v4',seed=BASE_SEED,outer_folds=3,outer_repeats=3,inner_folds=3,trials_per_family_per_strategy=50,families=FAMILIES,strategies=STRATEGIES,indicators=[False,True],data_sha256=meta['data_sha256'],versions=VERSIONS,baseline=BASE,
 selection='max mean inner AP corrected; tie 1e-6 min Brier then LR RF CatBoost XGBoost, then under_10 before under_5',primary='selection among undersampled real branches; all_real matched tuned control',threshold='max balanced accuracy inner OOF corrected; ties greater recall',
 strata='vascular group crossed with ages <60, 60..74, >=75; proportional survivor quotas with minimum one per nonempty stratum and largest remainder',undersampling='without replacement, all deaths retained; survivors min(available,ratio*deaths), ratios 10 and 5',preprocessing='fit imputation/scaling/encoder only on retained train',probabilities='r_s*p_raw/(1-p_raw+r_s*p_raw), r_s=train survivor inclusion rate by stratum; raw predictions retained; not guaranteed calibration',
 spaces={'LR':'C log .001..100','RF':'trees200..800 step100;depth2..8;leaf1,2,4,8,16;features sqrt,.5,1','CatBoost':'iterations100..800 step100;depth2..6;lr log .01...2;l2 log .01..100','XGBoost':'trees100..800 step100;depth1..5;lr log .01...2;child1..20;sub/col .6..1;lambda log .001..100;alpha0..5'})
CONFIG['target_author_clarification']=meta['target_author_clarification']
CONFIG['numeric_parser']='Excel 1900 serial restored for date-like numerical cells; five audited repairs'
CONFIG['endpoint_version']='questionnaire_16.09.2026_confirmed_23.10.2026'
CONFIG['target_author_clarification']=meta['target_author_clarification']
CONFIG['numeric_parser']='Excel 1900 serial restored for date-like numerical cells; five audited repairs'
CONFIG['endpoint_version']='questionnaire_16.09.2026_confirmed_23.10.2026'
current=__import__('nbformat').read(OUT/'02_модели_Optuna_и_эксперименты.ipynb',as_version=4)
CONFIG['training_code_sha256']=hashlib.sha256('\n'.join(current.cells[i].source for i in [1,2,4,6,8]).encode()).hexdigest()
RUN_ID='hospital_updated55_v4_'+hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:12];CONFIG['run_id']=RUN_ID
savejson(CONFIG,'02_обновлённые_протокол.json');STORAGE='sqlite:///'+str(OUT/'02_обновлённые_Optuna.sqlite')
def log(message):
 line=f'{datetime.datetime.now(datetime.timezone.utc).isoformat()} {RUN_ID} {message}';print(line,flush=True)
 with (OUT/'02_обновлённые_журнал.txt').open('a') as f:f.write(line+'\n')
log('Real-only protocol fixed before computations')
SAMPLING_AUDIT=[];STRATUM_AUDIT=[];INNER_SPLITS=[]
def splits(a,b,g,seed):
    ss=list(StratifiedGroupKFold(3,shuffle=True,random_state=seed).split(a,b,g))
    for tr,te in ss:
        assert not set(g[tr])&set(g[te])
        assert len(np.unique(b[tr]))==len(np.unique(b[te]))==2
    return ss
OUTER=[];splitrows=[]
for repeat in range(3):
    for fold,(tr,te) in enumerate(splits(X,y,groups,BASE_SEED+repeat)):
        OUTER.append((repeat,fold,tr,te))
        for side,ids in [("train",tr),("test",te)]:
            splitrows.extend(dict(repeat=repeat,fold=fold,side=side,row_id=int(D.row_id.iloc[i]),group_id=int(groups[i])) for i in ids)
pd.DataFrame(splitrows).to_csv(OUT/"02_обновлённые_внешние_разбиения.csv",index=False)
display(pd.DataFrame([dict(repeat=r,fold=f,train_n=len(tr),train_deaths=int(y[tr].sum()),test_n=len(te),test_deaths=int(y[te].sum())) for r,f,tr,te in OUTER]))

assert len(D)==548 and int(y.sum())==55
assert all(not set(groups[tr])&set(groups[te]) for _,_,tr,te in OUTER)


## Стратифицированный отбор без возвращения, поправка вероятностей и Optuna

In [ ]:

PRE_CACHE={};FIT_COUNT=0

def stratum_keys(a):
 ages=np.where(a.age_years.isna(),-1,np.searchsorted([60,75],a.age_years.to_numpy(),side='right'))
 return np.array([f'{int(v)}_{int(age)}' for v,age in zip(a.vascular_group,ages)])

def select_real(a,b,strategy,seed):
 sk=stratum_keys(a);alive=np.flatnonzero(b==0);dead=np.flatnonzero(b==1)
 labels=sorted(set(sk[alive]));counts=np.array([sum(sk[alive]==s) for s in labels],int)
 if strategy=='all_real':chosen=np.arange(len(a));quota=counts.copy()
 else:
  target=min(len(alive),int(strategy.split('_')[1])*len(dead));assert target>=len(labels)
  quota=np.ones(len(labels),int);capacity=counts-1;remaining=target-len(labels)
  ideal=remaining*capacity/capacity.sum() if capacity.sum() else np.zeros(len(labels))
  add=np.floor(ideal).astype(int);quota+=add
  order=sorted(range(len(labels)),key=lambda j:(-(ideal[j]-add[j]),labels[j]))
  for j in order[:target-int(quota.sum())]:quota[j]+=1
  assert quota.sum()==target and np.all(quota<=counts) and np.all(quota>=1)
  rng=np.random.default_rng(seed)
  survivors=np.concatenate([rng.choice(alive[sk[alive]==s],int(q),replace=False) for s,q in zip(labels,quota)])
  chosen=np.sort(np.r_[dead,survivors])
 assert len(np.unique(chosen))==len(chosen) and set(dead)<=set(chosen)
 rates={s:float(q/n) for s,q,n in zip(labels,quota,counts)}
 for s in set(sk):rates.setdefault(s,1.)
 return chosen,rates,[(s,int(n),int(q)) for s,n,q in zip(labels,counts,quota)]

def transform_pre(pre,a):
 zn=pre['scaler'].transform(pre['num_imputer'].transform(a[pre['nums']]))
 zc=pre['cat_imputer'].transform(a[pre['cats']])
 flag=a[pre['columns']].isna().to_numpy(float) if pre['indicators'] else np.zeros((len(a),0))
 return np.c_[zn,pre['encoder'].transform(zc),flag]

def fitted_context(a,b,indicators,strategy,seed):
 key=(tuple(a.index),tuple(a.columns),indicators,strategy,seed)
 if key in PRE_CACHE:return PRE_CACHE[key]
 keep,rates,strata=select_real(a,b,strategy,seed);retained=a.iloc[keep];train_y=b[keep].copy()
 nums=[c for c in a if c in meta['continuous']];cats=[c for c in a if c not in nums]
 ni=SimpleImputer(strategy='median',keep_empty_features=True);ci=SimpleImputer(strategy='most_frequent',keep_empty_features=True)
 zn=ni.fit_transform(retained[nums]);zc=ci.fit_transform(retained[cats]);scaler=StandardScaler().fit(zn);zn=scaler.transform(zn)
 encoder=OneHotEncoder(categories=[[0.,1.,2.] if c=='vascular_group' else [0.,1.] for c in cats],handle_unknown='ignore',sparse_output=False).fit(zc)
 flag=retained.isna().to_numpy(float) if indicators else np.zeros((len(retained),0));z=np.c_[zn,encoder.transform(zc),flag]
 names=['numeric__'+c for c in nums]+['categorical__'+c for c in encoder.get_feature_names_out(cats)]+(['indicators__'+c for c in a.columns] if indicators else [])
 pre=dict(nums=nums,cats=cats,columns=list(a.columns),num_imputer=ni,cat_imputer=ci,scaler=scaler,encoder=encoder,indicators=indicators,feature_names=names,fit_row_ids=list(map(int,retained.index)),inclusion_rates=rates,strategy=strategy)
 assert np.isfinite(z).all() and len(names)==z.shape[1]
 assert np.array_equal(train_y,D.hospital_death.loc[retained.index].to_numpy())
 context=len(SAMPLING_AUDIT)
 SAMPLING_AUDIT.append(dict(context_id=context,seed=seed,strategy=strategy,indicators=indicators,train_row_ids=json.dumps(list(map(int,a.index))),retained_row_ids=json.dumps(list(map(int,retained.index))),train_n=len(b),train_deaths=int(sum(b)),fit_n=len(train_y),fit_deaths=int(sum(train_y)),fit_survivors=int(sum(train_y==0)),fit_mortality=float(train_y.mean()),synthetic_n=0))
 STRATUM_AUDIT.extend(dict(context_id=context,seed=seed,strategy=strategy,stratum=s,real_survivors=n,retained_survivors=q,inclusion_probability=q/n) for s,n,q in strata)
 PRE_CACHE[key]=(pre,z,train_y);return PRE_CACHE[key]
def make_model(family,params,strategy,b,seed):
    weighted=strategy=="class_weights";ratio=float(sum(b==0)/sum(b==1))
    if family=="LR":return LogisticRegression(**params,l1_ratio=0,solver="lbfgs",max_iter=3000,class_weight="balanced" if weighted else None,random_state=seed)
    if family=="RF":return RandomForestClassifier(**params,class_weight="balanced" if weighted else None,random_state=seed,n_jobs=2)
    if family=="CatBoost":return CatBoostClassifier(**params,loss_function="Logloss",verbose=False,allow_writing_files=False,thread_count=2,random_seed=seed,auto_class_weights="Balanced" if weighted else None)
    return XGBClassifier(**params,objective="binary:logistic",eval_metric="logloss",tree_method="hist",n_jobs=2,random_state=seed,scale_pos_weight=ratio if weighted else 1.0)

def train(a,b,family,cfg,seed):
    global FIT_COUNT
    pre,z,train_y=fitted_context(a,b,cfg['indicators'],cfg['strategy'],seed)
    model=make_model(family,cfg['params'],cfg['strategy'],b,seed)
    model.fit(z,train_y);FIT_COUNT+=1
    assert list(model.classes_)==[0,1]
    return dict(preprocessor=pre,model=model,features=list(a.columns),family=family,config=cfg,run_id=RUN_ID)
def predict_raw(artifact,a):
    p=artifact['model'].predict_proba(transform_pre(artifact['preprocessor'],a))[:,1]
    assert np.isfinite(p).all() and np.all((p>=0)&(p<=1))
    return p

def predict(artifact,a):
 raw=predict_raw(artifact,a).astype(float);pre=artifact['preprocessor']
 r=np.array([pre['inclusion_rates'].get(s,1.) for s in stratum_keys(a)])
 assert np.all((r>0)&(r<=1))
 p=r*raw/(1-raw+r*raw);assert np.isfinite(p).all();return p
def threshold(b,p):
    fpr,tpr,t=roc_curve(b,p,drop_intermediate=False)
    score=(tpr+1-fpr)/2;valid=np.isfinite(t)
    mx=score[valid].max();ids=np.flatnonzero(valid&np.isclose(score,mx,atol=1e-12,rtol=0))
    idx=ids[np.argmax(tpr[ids])];return float(t[idx])
def metrics(b,p,t):
    h=p>=t;tn,fp,fn,tp=confusion_matrix(b,h,labels=[0,1]).ravel()
    return dict(AP=average_precision_score(b,p),ROC_AUC=roc_auc_score(b,p),Brier=brier_score_loss(b,p),sensitivity=tp/(tp+fn),specificity=tn/(tn+fp),precision=precision_score(b,h,zero_division=0),F1=f1_score(b,h,zero_division=0),accuracy=accuracy_score(b,h),TN=int(tn),FP=int(fp),FN=int(fn),TP=int(tp),n=len(b),deaths=int(sum(b)),mean_probability=float(np.mean(p)))
def suggest(trial,family):
    if family=="LR":return {"C":trial.suggest_float("C",1e-3,100,log=True)}
    if family=="RF":return {"n_estimators":trial.suggest_int("n_estimators",200,800,step=100),"max_depth":trial.suggest_int("max_depth",2,8),"min_samples_leaf":trial.suggest_categorical("min_samples_leaf",[1,2,4,8,16]),"max_features":trial.suggest_categorical("max_features",["sqrt",0.5,1.0])}
    if family=="CatBoost":return {"iterations":trial.suggest_int("iterations",100,800,step=100),"depth":trial.suggest_int("depth",2,6),"learning_rate":trial.suggest_float("learning_rate",.01,.2,log=True),"l2_leaf_reg":trial.suggest_float("l2_leaf_reg",.01,100,log=True)}
    return {"n_estimators":trial.suggest_int("n_estimators",100,800,step=100),"max_depth":trial.suggest_int("max_depth",1,5),"learning_rate":trial.suggest_float("learning_rate",.01,.2,log=True),"min_child_weight":trial.suggest_float("min_child_weight",1,20),"subsample":trial.suggest_float("subsample",.6,1),"colsample_bytree":trial.suggest_float("colsample_bytree",.6,1),"reg_lambda":trial.suggest_float("reg_lambda",.001,100,log=True),"reg_alpha":trial.suggest_float("reg_alpha",0,5)}
def inner_predictions(a,b,g,family,cfg,ss,seed):
    p=np.full(len(b),np.nan);aps=[];bs=[]
    for k,(tr,va) in enumerate(ss):
        artifact=train(a.iloc[tr],b[tr],family,cfg,seed+k)
        p[va]=predict(artifact,a.iloc[va]);aps.append(average_precision_score(b[va],p[va]));bs.append(brier_score_loss(b[va],p[va]))
    assert np.isfinite(p).all()
    return p,float(np.mean(aps)),float(np.mean(bs))
def tune(a,b,g,family,ss,name,seed,trials=50,fixed=None):
    study_name=RUN_ID+"_"+name+"_"+family
    sampler=optuna.samplers.TPESampler(seed=seed,multivariate=False,constant_liar=False)
    study=optuna.create_study(study_name=study_name,storage=STORAGE,direction="maximize",sampler=sampler,load_if_exists=True)
    if not study.trials:
        options=[(fixed["strategy"],i) for i in [False,True]]
        for s,i in options:study.enqueue_trial({**BASE[family],"strategy":s,"indicators":i})
        study.set_user_attr("config",CONFIG);study.set_user_attr("features",list(a.columns))
    def objective(trial):
        try:
            cfg={"strategy":trial.suggest_categorical("strategy",STRATEGIES) if fixed is None else fixed["strategy"],"indicators":trial.suggest_categorical("indicators",[False,True]),"params":suggest(trial,family)}
            p,ap,br=inner_predictions(a,b,g,family,cfg,ss,seed)
            trial.set_user_attr("config",cfg);trial.set_user_attr("Brier",br)
            return ap
        except Exception as exc:
            trial.set_user_attr("error",repr(exc));raise
    done=len([t for t in study.trials if t.state!=optuna.trial.TrialState.WAITING])
    if done<trials:study.optimize(objective,n_trials=trials-done,n_jobs=1,catch=(Exception,))
    complete=[t for t in study.trials if t.state==optuna.trial.TrialState.COMPLETE]
    assert len(complete)==trials and len(study.trials)==trials,study_name
    maximum=max(t.value for t in complete);eligible=[t for t in complete if maximum-t.value<=1e-6]
    best=min(eligible,key=lambda t:(t.user_attrs["Brier"],t.number))
    cfg=best.user_attrs["config"]
    p,ap,br=inner_predictions(a,b,g,family,cfg,ss,seed)
    log(f"{name} {family}: {len(complete)}/{trials} complete; inner AP={ap:.5f}; {cfg}")
    return {"family":family,"config":cfg,"inner_AP":ap,"inner_Brier":br,"threshold":threshold(b,p),"study_name":study_name,"best_trial":best.number,"completed":len(complete)}

def choose(results):
 mx=max(r['inner_AP'] for r in results)
 return min([r for r in results if mx-r['inner_AP']<=1e-6],key=lambda r:(r['inner_Brier'],FAMILIES.index(r['family']),STRATEGIES.index(r['config']['strategy'])))
def records(ids,r,f,label,p,raw,t,cfg):
 return [dict(row_id=int(D.row_id.iloc[j]),group_id=int(groups[j]),repeat=r,fold=f,family=label,y_true=int(y[j]),p_death=float(prob),p_raw=float(pr),threshold=float(t),run_id=RUN_ID,configuration=json.dumps(cfg,sort_keys=True)) for j,prob,pr in zip(ids,p,raw)]
for strategy in STRATEGIES:
 keep,rates,_=select_real(X,y,strategy,BASE_SEED)
 assert set(np.flatnonzero(y==1))<=set(keep)
 assert np.array_equal(X.iloc[keep].to_numpy(),X.loc[X.index[keep]].to_numpy(),equal_nan=True)
 assert np.array_equal(keep,select_real(X,y,strategy,BASE_SEED)[0])
 if strategy!='all_real':assert sum(y[keep]==0)==min(sum(y==0),int(strategy.split('_')[1])*sum(y==1))
print('Subset checks passed: real rows only, all deaths retained, deterministic.')


## Общая внешняя проверка

In [ ]:

OOF=[];RESULTS=[];FOLD_METRICS=[];RAW_METRICS=[]
for repeat,fold,tr,te in OUTER:
 seed=BASE_SEED+100*repeat+10*fold;a=X.iloc[tr];b=y[tr];g=groups[tr];ss=splits(a,b,g,seed+1000)
 for k,(it,iv) in enumerate(ss):
  INNER_SPLITS.extend(dict(repeat=repeat,fold=fold,inner_fold=k,side=side,row_id=int(D.row_id.iloc[tr[j]]),group_id=int(g[j])) for side,ii in [('train',it),('validation',iv)] for j in ii)
 candidates=[];probs={}
 for family in FAMILIES:
  for strategy in STRATEGIES:
   result=tune(a,b,g,family,ss,f'{strategy}_r{repeat}_f{fold}',seed,fixed=dict(strategy=strategy))
   candidates.append(result);RESULTS.append(dict(repeat=repeat,fold=fold,**result))
   model=train(a,b,family,result['config'],seed);pp=predict(model,X.iloc[te]);raw=predict_raw(model,X.iloc[te]);tt=result['threshold'];label=family+'__'+strategy
   probs[(family,strategy)]=(pp,raw)
   OOF.extend(records(te,repeat,fold,label,pp,raw,tt,result['config']))
   FOLD_METRICS.append(dict(repeat=repeat,fold=fold,family=label,**metrics(y[te],pp,tt)))
   RAW_METRICS.append(dict(repeat=repeat,fold=fold,family=label,AP=average_precision_score(y[te],raw),ROC_AUC=roc_auc_score(y[te],raw),Brier=brier_score_loss(y[te],raw),mean_probability=float(raw.mean())))
  cfg=dict(params=BASE[family],strategy='all_real',indicators=True)
  ip,_,_=inner_predictions(a,b,g,family,cfg,ss,seed);tb=threshold(b,ip)
  baseline=train(a,b,family,cfg,seed);pb=predict(baseline,X.iloc[te]);rb=predict_raw(baseline,X.iloc[te])
  OOF.extend(records(te,repeat,fold,family+'_baseline',pb,rb,tb,cfg))
  FOLD_METRICS.append(dict(repeat=repeat,fold=fold,family=family+'_baseline',**metrics(y[te],pb,tb)))
  winner=choose([r for r in candidates if r['family']==family and r['config']['strategy']!='all_real'])
  pp,raw=probs[(family,winner['config']['strategy'])]
  OOF.extend(records(te,repeat,fold,family,pp,raw,winner['threshold'],winner['config']))
  FOLD_METRICS.append(dict(repeat=repeat,fold=fold,family=family,chosen_strategy=winner['config']['strategy'],**metrics(y[te],pp,winner['threshold'])))
 for label,subset in [('selector',[r for r in candidates if r['config']['strategy']!='all_real']),('selector_all_real',[r for r in candidates if r['config']['strategy']=='all_real'])]:
  winner=choose(subset);pp,raw=probs[(winner['family'],winner['config']['strategy'])]
  cfg=dict(selected_family=winner['family'],**winner['config'])
  OOF.extend(records(te,repeat,fold,label,pp,raw,winner['threshold'],cfg))
  FOLD_METRICS.append(dict(repeat=repeat,fold=fold,family=label,chosen_family=winner['family'],chosen_strategy=winner['config']['strategy'],**metrics(y[te],pp,winner['threshold'])))
 prior=float(b.mean());pp=np.full(len(te),prior)
 OOF.extend(records(te,repeat,fold,'prevalence_baseline',pp,pp,1.,dict(train_prevalence=prior)))
 tab=D.iloc[tr].groupby('vascular_group').hospital_death.agg(['sum','size']);rates=(tab['sum']+2*prior)/(tab['size']+2)
 pp=D.iloc[te].vascular_group.map(rates).fillna(prior).to_numpy()
 OOF.extend(records(te,repeat,fold,'vascular_baseline',pp,pp,1.,dict(smoothing_strength=2)))
 for label in ['prevalence_baseline','vascular_baseline']:
  vv=pd.DataFrame(OOF);vv=vv[(vv.family==label)&(vv.repeat==repeat)&(vv.fold==fold)]
  FOLD_METRICS.append(dict(repeat=repeat,fold=fold,family=label,**metrics(vv.y_true.to_numpy(),vv.p_death.to_numpy(),1.)))
 log(f'Outer r{repeat} f{fold} complete')
 pd.DataFrame(OOF).to_csv(OUT/'02_обновлённые_oof.csv',index=False)
 pd.DataFrame(FOLD_METRICS).to_csv(OUT/'02_обновлённые_метрики_блоков.csv',index=False)
 pd.DataFrame(RAW_METRICS).to_csv(OUT/'02_обновлённые_сырые_метрики.csv',index=False)
 savejson(RESULTS,'02_обновлённые_конфигурации_блоков.json')
 pd.DataFrame(SAMPLING_AUDIT).to_csv(OUT/'02_обновлённые_аудит_отбора.csv',index=False)
 pd.DataFrame(STRATUM_AUDIT).to_csv(OUT/'02_обновлённые_аудит_страт.csv',index=False)
pd.DataFrame(INNER_SPLITS).to_csv(OUT/'02_обновлённые_внутренние_разбиения.csv',index=False)
oof=pd.DataFrame(OOF);fm=pd.DataFrame(FOLD_METRICS)
assert oof.groupby(['repeat','family']).row_id.nunique().eq(len(D)).all()
assert not oof.duplicated(['repeat','family','row_id']).any()
display(fm.groupby('family')[['AP','ROC_AUC','Brier','sensitivity']].mean())


## Финальный внутренний выбор и обучение сокращённой модели

In [ ]:

final_ss=splits(X,y,groups,BASE_SEED+9999)
FINAL_RESULTS=[tune(X,y,groups,f,final_ss,'final_'+s,BASE_SEED+9999,fixed=dict(strategy=s)) for f in FAMILIES for s in STRATEGIES]
FINAL=choose([r for r in FINAL_RESULTS if r['config']['strategy']!='all_real']);FINAL_CONTROL=choose([r for r in FINAL_RESULTS if r['config']['strategy']=='all_real'])
artifact=train(X,y,FINAL['family'],FINAL['config'],BASE_SEED+9999)
artifact.update(threshold=FINAL['threshold'],source_data_sha256=meta['data_sha256'],versions=VERSIONS,probability_rule=CONFIG['probabilities'])
joblib.dump(artifact,OUT/'02_обновлённые_final_model.joblib')
savejson(dict(run_id=RUN_ID,selection=FINAL,all_real_control_selection=FINAL_CONTROL,all_families=FINAL_RESULTS,features=FEATURES,interpretation='balanced real-data artifact; independent quality belongs to nested selector'),'02_обновлённые_final_config.json')
rows=[]
for k,(it,iv) in enumerate(final_ss):rows.extend(dict(inner_fold=k,side=side,row_id=int(D.row_id.iloc[j]),group_id=int(groups[j])) for side,ids in [('train',it),('validation',iv)] for j in ids)
pd.DataFrame(rows).to_csv(OUT/'02_обновлённые_финальные_разбиения.csv',index=False)
print('FINAL BALANCED',FINAL);print('ALL REAL CONTROL',FINAL_CONTROL)
display(pd.DataFrame(FINAL_RESULTS)[['family','inner_AP','inner_Brier','threshold','best_trial','config']])


## Парные сравнения и калибровка

In [ ]:

def summarize_oof(frame):
 rows=[dict(name=name,repeat=r,**metrics(v.y_true.to_numpy(),v.p_death.to_numpy(),v.threshold.to_numpy())) for (name,r),v in frame.groupby(['family','repeat'])]
 reps=pd.DataFrame(rows);cols=['AP','ROC_AUC','Brier','sensitivity','specificity','precision','F1','accuracy','mean_probability']
 return reps,reps.groupby('name')[cols].mean()
repeats,summary=summarize_oof(oof);repeats.to_csv(OUT/'02_обновлённые_метрики_повторений.csv',index=False);summary.to_csv(OUT/'02_обновлённые_сводка_повторений.csv')
foldsummary=fm.groupby('family')[['AP','ROC_AUC','Brier','sensitivity','specificity','precision','F1','mean_probability']].mean();foldsummary.to_csv(OUT/'02_обновлённые_сводка_блоков.csv')
rawrows=[dict(family=name,repeat=r,AP=average_precision_score(v.y_true,v.p_raw),ROC_AUC=roc_auc_score(v.y_true,v.p_raw),Brier=brier_score_loss(v.y_true,v.p_raw),mean_probability=v.p_raw.mean()) for (name,r),v in oof.groupby(['family','repeat'])]
rawsummary=pd.DataFrame(rawrows).groupby('family')[['AP','ROC_AUC','Brier','mean_probability']].mean();rawsummary.to_csv(OUT/'02_обновлённые_сырые_сводка_повторений.csv')
names=[f+'__'+s for f in FAMILIES for s in STRATEGIES]+['selector','selector_all_real']
arrays={name:np.array([oof[(oof.family==name)&(oof.repeat==r)].set_index('row_id').loc[D.row_id,'p_death'].to_numpy() for r in range(3)]) for name in names}
thresholds={name:np.array([oof[(oof.family==name)&(oof.repeat==r)].set_index('row_id').loc[D.row_id,'threshold'].to_numpy() for r in range(3)]) for name in names}
unique=np.unique(groups);bygroup={g:np.flatnonzero(groups==g) for g in unique};rng=np.random.default_rng(BASE_SEED+4444)
boot={name:[] for name in names}
for iteration in range(1000):
 ids=np.concatenate([bygroup[g] for g in rng.choice(unique,len(unique),replace=True)])
 if len(np.unique(y[ids]))<2:continue
 yy=y[ids]
 for name,pp in arrays.items():
  scores=[]
  for r in range(3):
   pr=pp[r,ids];hh=pr>=thresholds[name][r,ids]
   scores.append([average_precision_score(yy,pr),roc_auc_score(yy,pr),brier_score_loss(yy,pr),float(hh[yy==1].mean())])
  boot[name].append(np.mean(scores,axis=0))
 if iteration%250==0:log(f'Bootstrap {iteration}/1000')
boot={k:np.asarray(v) for k,v in boot.items()};ci=[]
for name,ss in boot.items():
 for j,metric in enumerate(['AP','ROC_AUC','Brier','sensitivity']):
  lo,hi=np.quantile(ss[:,j],[.025,.975]);ci.append(dict(family=name,metric=metric,estimate=summary.loc[name,metric],lower=lo,upper=hi,scope='group bootstrap fixed OOF; means of full repeats'))
pd.DataFrame(ci).to_csv(OUT/'02_обновлённые_bootstrap_CI.csv',index=False)
comparisons=[(f+'__'+s,f+'__all_real') for f in FAMILIES for s in ['under_10','under_5']]+[('selector','selector_all_real')];delta=[]
for plus,minus in comparisons:
 for j,metric in enumerate(['AP','ROC_AUC','Brier','sensitivity']):
  vv=boot[plus][:,j]-boot[minus][:,j];lo,hi=np.quantile(vv,[.025,.975])
  delta.append(dict(plus=plus,minus=minus,metric=metric,delta=summary.loc[plus,metric]-summary.loc[minus,metric],lower=lo,upper=hi,scope='paired fixed OOF group bootstrap; no multiplicity correction'))
delta=pd.DataFrame(delta);delta.to_csv(OUT/'02_обновлённые_парные_изменения.csv',index=False)
cal=[]
for (name,r),v in oof.groupby(['family','repeat']):
 v=v.copy();v['bin']=pd.qcut(v.p_death,5,duplicates='drop')
 for bn,q in v.groupby('bin',observed=True):cal.append(dict(family=name,repeat=r,bin=str(bn),n=len(q),deaths=int(q.y_true.sum()),mean_prediction=q.p_death.mean(),event_rate=q.y_true.mean()))
cal=pd.DataFrame(cal);cal.to_csv(OUT/'02_обновлённые_калибровка.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(13,5))
for name in ['selector','selector_all_real']:
 vv=cal[cal.family==name];axes[0].plot(vv.mean_prediction,vv.event_rate,'o',label=name)
 for rr,g in vv.groupby('repeat'):axes[0].plot(g.mean_prediction,g.event_rate,alpha=.25)
axes[0].plot([0,1],[0,1],'--',color='grey');axes[0].set(xlabel='Средний прогноз смерти',ylabel='Наблюдаемая смертность',title='Независимая проверка общей процедуры');axes[0].legend()
for j,s in enumerate(STRATEGIES):axes[1].bar(np.arange(4)+(j-1)*.24,[summary.loc[f+'__'+s,'AP'] for f in FAMILIES],width=.24,label=s)
axes[1].set_xticks(np.arange(4),FAMILIES);axes[1].set(ylabel='AP смерти',title='Среднее трёх полных OOF-повторений');axes[1].legend();figsave('02_обновлённые_качество_калибровка')
trialframes=[]
for study in optuna.get_all_study_summaries(storage=STORAGE):
 if study.study_name.startswith(RUN_ID):
  ss=optuna.load_study(study_name=study.study_name,storage=STORAGE);tt=ss.trials_dataframe();tt.insert(0,'study',study.study_name);trialframes.append(tt)
trials_df=pd.concat(trialframes,ignore_index=True);trials_df.to_csv(OUT/'02_обновлённые_trials.csv',index=False)
pd.DataFrame(SAMPLING_AUDIT).to_csv(OUT/'02_обновлённые_аудит_отбора.csv',index=False);pd.DataFrame(STRATUM_AUDIT).to_csv(OUT/'02_обновлённые_аудит_страт.csv',index=False)
savejson(dict(run_id=RUN_ID,protocol=CONFIG,data_sha256=sha(OUT/'01_данные.csv'),model_sha256=sha(OUT/'02_обновлённые_final_model.joblib'),model_fits_current_execution=FIT_COUNT,unique_training_contexts=len(SAMPLING_AUDIT),studies=len(trialframes),trial_states=trials_df.state.value_counts().to_dict(),completed_utc=datetime.datetime.now(datetime.timezone.utc).isoformat()),'02_обновлённые_происхождение.json')
log('All real-only nested experiments completed');display(foldsummary);display(summary);display(delta)


## Фактический отчёт

In [ ]:

cols=['AP','ROC_AUC','Brier','sensitivity','specificity','precision']
retained=artifact['preprocessor']['fit_row_ids'];ry=D.loc[retained,'hospital_death']
subgroups=D.groupby('vascular_group').hospital_death.agg(n='size',deaths='sum')
family_sections=[]
for f in FAMILIES:
 cfgrows=pd.DataFrame([dict(strategy=r['config']['strategy'],indicators=r['config']['indicators'],inner_AP=r['inner_AP'],inner_Brier=r['inner_Brier'],threshold=r['threshold'],params=json.dumps(r['config']['params'],ensure_ascii=False)) for r in FINAL_RESULTS if r['family']==f])
 labels=[f+'__'+s for s in STRATEGIES]+[f,f+'_baseline']
 family_sections.append(f'''## {f}

Пространство: {CONFIG['spaces'][f]}. По 50 trials на ветвь × 3 ветви × 10 контекстов (9 внешних обучений и финальный поиск) = 1500 trials на семейство. Настроенные показатели оценивают весь пайплайн, а не изолированный эффект гиперпараметров. Baseline: {json.dumps(BASE[f])}, all_real, индикаторы True.

Финальный внутренний поиск (не независимое качество):

{cfgrows.to_markdown(index=False,floatfmt='.5f')}

Средние девяти внешних блоков:

{foldsummary.loc[labels,cols].to_markdown(floatfmt='.5f')}

Средние трёх полных OOF-повторений:

{summary.loc[labels,cols].to_markdown(floatfmt='.5f')}

Калибровка для этого семейства, baseline и фиксированных ветвей сохранена с n и числом смертей в каждом бине в «02_обновлённые_калибровка.csv». 55 отметок относятся к 51 группе с исходом, поэтому оценка малых различий ограничена. Все параметры и независимые прогнозы доступны в общих CSV.
''')
flagfreq=pd.DataFrame([dict(family=r['family'],strategy=r['config']['strategy'],indicators=r['config']['indicators']) for r in RESULTS]).value_counts().rename('outer_blocks').reset_index()
flagfreq.to_csv(OUT/'02_обновлённые_частота_индикаторов.csv',index=False)
protocol_notes=f'''Пять датоподобных чисел исходного CSV восстановлены из формата Excel; три входят в новую когорту, две относятся к исключённой строке. Предварительный технический поиск прерван и сохранён отдельно, ни один его trial не включён в текущий поиск. Госпитальный исход обновлён по листу «анкета» согласно уточнению автора 23.10.2026. {len(D)} записей, {sum(y)} отметок смерти ({100*y.mean():.2f}%), {len(np.unique(groups))} групп. 55 отметок находятся в 51 группе; все повторные записи удержаны вместе. Новые строки не генерируются, веса классов отсутствуют. Исходные 14 предикторов сохранены; пропуск возраста допустим и образует отдельный слой отбора.

Сохранены базовый seed 2026, алгоритм разбиения и поиск: 3 внешних групповых стратифицированных блока × 3 повторения, внутри каждого обучения 3 блока, Optuna TPE(seed, multivariate=False, constant_liar=False), по 50 trials на семейство и ветвь. Первые две попытки — фиксированные baseline-параметры с индикаторами False/True, входят в бюджет 50. Затем адаптивный поиск. Trials последовательные, модели используют 2 потока, early stopping отсутствует. Разбиения построены заново: состав и разметка изменились, прежние точные индексы не сохраняются. Старые trials не используются.

Ветви: all_real, under_10 и under_5. Все смерти сохраняются; выжившие отбираются без возвращения по бассейну × возрасту <60/60–74/≥75/неизвестен. Квоты пропорциональные, минимум одна запись на непустой слой, остаток по наибольшим дробным частям. Размер выживших = min(доступно, отношение × число смертей). Поэтому under_10 на полной когорте не сокращает выборку: исходное отношение уже меньше 10:1; фактический отбор каждого CV-контекста сохранён в аудите. under_5 повышает долю смертей до 1/6 при достаточном числе выживших.

Медианы, моды, StandardScaler и one-hot обучаются только на оставленных train-записях. Индикаторы всех 14 исходных полей, включая СРБ, вычисляются до импутации; False означает отсутствие всех индикаторов. Числовой СРБ сохранён. Полностью пустой столбец сохраняется с нулевой заменой. Ни одна проверочная запись не участвует в этих оценках.

После сокращения используется прежняя поправка odds по обучающей доле сохранённых выживших r_s: p=r_s*p_raw/(1-p_raw+r_s*p_raw). В all_real и при отсутствии сокращения r_s=1. Это не гарантия калибровки; сырые выходы сохранены. Критерий Optuna — максимальный средний AP смерти трёх внутренних блоков по скорректированным прогнозам, равенство 1e-6 разрешается меньшим Brier. При выборе семейства далее фиксированный порядок LR/RF/CatBoost/XGBoost, затем under_10 перед under_5. Порог — максимальная balanced accuracy внутренних OOF, при равенстве больший recall.

Основной selector выбирает среди under_10/under_5 четырёх семейств; selector_all_real — отдельный контроль четырёх all_real. Это прежнее правило, RF заранее не фиксирован. Во внешних блоках исходная смертность сохраняется. Всего {len(trials_df)} COMPLETE trials в {len(trialframes)} studies. Все фактические seeds, разбиения, модели и OOF связаны RUN_ID {RUN_ID}.
'''
report=f'''# Четыре модели и Optuna: обновлённые госпитальные исходы

Статус: выполнено. RUN_ID {RUN_ID}. Исполнено в чистом ядре Python (Analytics).

## Методы

{protocol_notes}

## Оценка полной процедуры

Средние девяти внешних блоков:

{foldsummary.loc[FAMILIES+['selector','selector_all_real','prevalence_baseline'],cols].to_markdown(floatfmt='.5f')}

Средние трёх полных OOF-повторений:

{summary.loc[FAMILIES+['selector','selector_all_real','prevalence_baseline'],cols].to_markdown(floatfmt='.5f')}

AP — Average Precision, не трапецеидальная PR-AUC; положительный класс смерть=1. Brier меньше — лучше. Recall=TP/(TP+FN), specificity=TN/(TN+FP), precision=TP/(TP+FP). Пороги различаются между внешними обучениями и выбираются без внешних исходов. AP/AUC объединённых прогнозов разных fitted моделей могут отличаться от среднего отдельных блоков. Три повторения не увеличивают число независимых пациентов и смертей.

{''.join(family_sections)}

## Пропуски и балансировка

Индикаторы True/False испытывались в каждой ветви и настраивались вместе с параметрами. Частоты выбора:

{flagfreq.to_markdown(index=False)}

Это частоты внутреннего выбора, не отдельная причинная оценка эффекта индикаторов. Фиксированные ветви all_real/under_10/under_5 имеют одинаковые бюджеты и разбиения. Парное сравнение по 1000 bootstrap-выборкам групп:

{delta.to_markdown(index=False,floatfmt='.5f')}

Интервалы условны на фиксированных OOF без переобучения; множественные сравнения не скорректированы. Группы, их госпитализации и повторения ресэмплируются совместно; пары используют одинаковые выборки.

Сырые вероятности (средние полных повторений):

{rawsummary.loc[['selector','selector_all_real']+[f+'__'+s for f in FAMILIES for s in STRATEGIES]].to_markdown(floatfmt='.5f')}

![Качество и калибровка](02_обновлённые_качество_калибровка.png)

## Окончательная модель

{json.dumps(FINAL,ensure_ascii=False)}

Обучено на {len(retained)} реальных записях, {int(ry.sum())} отметок смерти и {int((ry==0).sum())} выживших. Финальный внутренний AP не объявляется независимой оценкой. Качество процедуры выбора — внешние прогнозы selector. Контроль all_real: {json.dumps(FINAL_CONTROL,ensure_ascii=False)}.

## Ограничения и материал для статьи

Обновление разметки принято по уточнению автора; противоречия со старым полем, годы смерти и повторные летальные отметки требуют клинической проверки. Старое и новое исследования отличаются исходом, когортой и разбиениями: разность их метрик описательная, не парная оценка эффекта увеличения числа смертей. Сокращение не добавляет событий и может уменьшать информацию о выживших. Индикаторы могут отражать процесс обследования. Пол и группы пациентов получены по ФИО, единицы отдельных лабораторных полей не подтверждены. Автор сообщил доступность признаков при поступлении 08.10.2026; временные метки независимо не проверены. Новая клиническая внешняя валидация отсутствует.

Для статьи: «Четыре семейства классификаторов настроены во вложенной групповой кросс-валидации с одинаковым бюджетом 50 trials на ветвь. Предобработка и сокращение большинства выполнялись внутри обучения, внешние части сохраняли естественную частоту исхода. Отдельно оценена вся процедура выбора семейства, порога и отношения отбора».

Воспроизведение: 01 → 02 → 03, чистые ядра analytics_learning / Python (Analytics). [Протокол](02_обновлённые_протокол.json), [trials](02_обновлённые_trials.csv), [OOF](02_обновлённые_oof.csv), [параметры](02_обновлённые_final_config.json), [модель](02_обновлённые_final_model.joblib), [происхождение](02_обновлённые_происхождение.json). Прежние результаты находятся в проверенном архиве.
'''
(OUT/'02_модели_Optuna_и_эксперименты.md').write_text(report)
print('Report02 saved',RUN_ID)


## Интерпретация сохранённых и независимо проверенных результатов

Качество обновлённой разметки ограничено. Средние трёх полных OOF-повторений процедуры selector: AP 0,12941, ROC-AUC 0,58054, Brier 0,09332, recall смерти 72,73%, специфичность 47,19%, precision 13,33%. При контроле только по сосудистому бассейну AP 0,13144, ROC-AUC 0,62459 и Brier 0,08780; Brier постоянного контроля 0,09030. По этим точечным оценкам процедура ML не превзошла бассейновый контроль, а её вероятностная ошибка больше постоянного контроля. Это не утверждение статистически доказанного превосходства контроля: парный интервал именно этого сравнения отдельно не рассчитывался. Высокий recall при низкой специфичности и precision не подтверждает пригодность для клинических решений.

Относительно настроенной процедуры all_real сокращённая процедура дала ΔAP +0,01060 с условным 95% интервалом [−0,00649; 0,03690], включающим ноль. Brier уменьшился на 0,00422, условный интервал [−0,00650; −0,00199], но это не означает преимущества над простыми контролями. Финальный XGBoost выбран по заранее принятому внутреннему AP, а не по внешней таблице; внешний результат сохранён без попытки подобрать более выгодное правило после просмотра проверки. Интервалы относятся к фиксированным OOF и не включают всю неопределённость переобучения. Эти результаты не поддерживают утверждение о надёжном клиническом прогнозе всех 55 отметок.

Временная проверка остаётся существенной: у 10 записей год смерти на 1–7 лет позже года операции. Все 55 отметок сохранены по авторскому уточнению, но госпитальный срок событий независимо не подтверждён; переносить результат в статью как проверенный прогноз смерти в пределах индексной госпитализации без клинической сверки нельзя.